# Italy Road Accident Analysis
**Analyst:** Juan Xavier Gomez Illingworth

**NB:** Parts of the code were constructed with the assistance of Gen AI.

In [10]:
# Load libraries and configure Altair
import pandas as pd
import numpy as np
import altair as alt
import urllib.request
import json
import os

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [11]:
# Load and clean road accident casualty data
ki_road_accidents = pd.read_csv('../data/Killed and injured in road accidents (IT1,41_270_DF_DCIS_MORTIFERITISTR1_1,1.0).csv')
ki_road_accidents['Territory'] = ki_road_accidents['Territory'].str.replace("'", "", regex=False).str.replace('"', "'", regex=False).str.replace("�", "é", regex=False)
ki_road_accidents.head()

,FREQ,Frequency,REF_AREA,Territory,DATA_TYPE,Indicator,ACCIDENT_LOCALIZATON,Localization of the accident,INTERSECTION,Intersection (DESC),...,PERSON_CLASS,Person class,AGE,Age (DESC),SEX,Sex (DESC),MONTH,Month (DESC),TIME_PERIOD,Observation
0,A,Annual,ITC1,Piemonte,KILLINJ,Killed and injured,9,Total,9,Total,...,C,Driver,TOTAL,Total,9,Total,99,Total,2010,233
1,A,Annual,ITC1,Piemonte,KILLINJ,Killed and injured,9,Total,9,Total,...,C,Driver,TOTAL,Total,9,Total,99,Total,2011,206
2,A,Annual,ITC1,Piemonte,KILLINJ,Killed and injured,9,Total,9,Total,...,C,Driver,TOTAL,Total,9,Total,99,Total,2012,219
3,A,Annual,ITC1,Piemonte,KILLINJ,Killed and injured,9,Total,9,Total,...,C,Driver,TOTAL,Total,9,Total,99,Total,2013,184
4,A,Annual,ITC1,Piemonte,KILLINJ,Killed and injured,9,Total,9,Total,...,C,Driver,TOTAL,Total,9,Total,99,Total,2014,179


In [12]:
# Load population files and combine
pop_df1 = pd.read_csv('../data/Italy, regions, provinces (IT1,22_289_DF_DCIS_POPRES1_1,1.0).csv')
pop_df2 = pd.read_csv('../data/Italy, regions, provinces (IT1,22_289_DF_DCIS_POPRES1_1,1.0) (1).csv')
population_df = pd.concat([pop_df1, pop_df2], ignore_index=True)
population_df = population_df.sort_values(['REF_AREA', 'TIME_PERIOD']).reset_index(drop=True)
population_df.head()

,FREQ,Frequency,REF_AREA,Territory,DATA_TYPE,Indicator,SEX,Gender,AGE,Age (DESC),MARITAL_STATUS,Marital status,TIME_PERIOD,Observation,OBS_STATUS,Observation status
0,A,Annual,ITC1,Piemonte,JAN,Population on 1st January,9,Total,TOTAL,Total,99,Total,2019,4328565,NaN,NaN
1,A,Annual,ITC1,Piemonte,JAN,Population on 1st January,9,Total,TOTAL,Total,99,Total,2020,4311217,NaN,NaN
2,A,Annual,ITC1,Piemonte,JAN,Population on 1st January,9,Total,TOTAL,Total,99,Total,2021,4274945,NaN,NaN
3,A,Annual,ITC1,Piemonte,JAN,Population on 1st January,9,Total,TOTAL,Total,99,Total,2022,4256350,NaN,NaN
4,A,Annual,ITC1,Piemonte,JAN,Population on 1st January,9,Total,TOTAL,Total,99,Total,2023,4251351,NaN,NaN


In [13]:
# Prepare casualty records filtered to relevant dimensions
chart_data = ki_road_accidents[
    (ki_road_accidents['Result (DESC)'].isin(['Killed', 'Injured'])) &
    (ki_road_accidents['Person class'].isin(['Driver', 'Passenger', 'Pedestrian'])) &
    (ki_road_accidents['Age (DESC)'] == 'Total') &
    (ki_road_accidents['Sex (DESC)'] == 'Total') &
    (ki_road_accidents['Month (DESC)'] == 'Total') &
    (ki_road_accidents['Intersection (DESC)'] == 'Total') &
    (ki_road_accidents['Localization of the accident'] == 'Total') &
    (ki_road_accidents['Road accident type'] == 'Total')
].copy()
grouped_data = chart_data.groupby(['TIME_PERIOD','Result (DESC)','Person class','Territory'])['Observation'].sum().reset_index()
all_regions_data = grouped_data[~grouped_data['Territory'].isin(['Provincia Autonoma Bolzano / Bozen', 'Provincia Autonoma Trento'])].groupby(['TIME_PERIOD','Result (DESC)','Person class'])['Observation'].sum().reset_index()
all_regions_data['Territory'] = 'All Regions'
final_data = pd.concat([grouped_data, all_regions_data], ignore_index=True)
final_data.head()

,TIME_PERIOD,Result (DESC),Person class,Territory,Observation
0,2010,Injured,Driver,Abruzzo,4338
1,2010,Injured,Driver,Basilicata,1192
2,2010,Injured,Driver,Calabria,3560
3,2010,Injured,Driver,Campania,10784
4,2010,Injured,Driver,Emilia-Romagna,20034


In [14]:
# Prepare and backfill population totals
pop_filtered = population_df[(population_df['Gender'] == 'Total') & (population_df['Age (DESC)'] == 'Total') & (population_df['Marital status'] == 'Total')][['Territory', 'TIME_PERIOD', 'Observation']].copy()
pop_filtered = pop_filtered.rename(columns={'Observation': 'Population'})
pop_2019 = pop_filtered[pop_filtered['TIME_PERIOD'] == 2019].copy()
backfilled_df = pd.concat([pop_2019.assign(TIME_PERIOD=year) for year in range(2010, 2019)], ignore_index=True)
pop_filtered = pd.concat([backfilled_df, pop_filtered], ignore_index=True).sort_values(['Territory', 'TIME_PERIOD']).reset_index(drop=True)
italy_total_pop = pop_filtered[pop_filtered['Territory'] != 'Trentino Alto Adige / Südtirol'].groupby('TIME_PERIOD')['Population'].sum().reset_index()
italy_total_pop['Territory'] = 'All Regions'
population_data = pd.concat([pop_filtered, italy_total_pop], ignore_index=True)
population_data.head()

,Territory,TIME_PERIOD,Population
0,"'Valle d""'Aosta / Vallée d""'Aoste'",2010,125653
1,"'Valle d""'Aosta / Vallée d""'Aoste'",2011,125653
2,"'Valle d""'Aosta / Vallée d""'Aoste'",2012,125653
3,"'Valle d""'Aosta / Vallée d""'Aoste'",2013,125653
4,"'Valle d""'Aosta / Vallée d""'Aoste'",2014,125653


In [15]:
# Review prepared population data
population_data.head()

,Territory,TIME_PERIOD,Population
0,"'Valle d""'Aosta / Vallée d""'Aoste'",2010,125653
1,"'Valle d""'Aosta / Vallée d""'Aoste'",2011,125653
2,"'Valle d""'Aosta / Vallée d""'Aoste'",2012,125653
3,"'Valle d""'Aosta / Vallée d""'Aoste'",2013,125653
4,"'Valle d""'Aosta / Vallée d""'Aoste'",2014,125653


In [16]:
# Merge casualties with population and compute metrics
enhanced_data = final_data.merge(population_data, on=['Territory', 'TIME_PERIOD'], how='left')
enhanced_data['Per_100k'] = (enhanced_data['Observation'] / enhanced_data['Population']) * 100000
count_data = enhanced_data.copy()
count_data['Metric_Type'] = 'Total Counts'
count_data['Value'] = count_data['Observation']
rate_data = enhanced_data[enhanced_data['Population'].notna()].copy()
rate_data['Metric_Type'] = 'Per 100,000 Inhabitants'
rate_data['Value'] = rate_data['Per_100k']
chart_data_enhanced = pd.concat([count_data, rate_data], ignore_index=True)
injured_and_killed = chart_data_enhanced[chart_data_enhanced['Result (DESC)'].isin(['Injured', 'Killed'])].copy()
combined_data = injured_and_killed.groupby(['Territory','TIME_PERIOD','Person class','Metric_Type']).agg({'Value': 'sum','Observation': 'sum','Population': 'first'}).reset_index()
combined_data['Result (DESC)'] = 'Injured + Killed'
chart_data_enhanced = pd.concat([chart_data_enhanced, combined_data], ignore_index=True)
chart_data_enhanced.head()

,TIME_PERIOD,Result (DESC),Person class,Territory,Observation,Population,Per_100k,Metric_Type,Value
0,2010,Injured,Driver,Abruzzo,4338,1300645.0,333.526827,Total Counts,4338.0
1,2010,Injured,Driver,Basilicata,1192,558587.0,213.395586,Total Counts,1192.0
2,2010,Injured,Driver,Calabria,3560,1912021.0,186.190424,Total Counts,3560.0
3,2010,Injured,Driver,Campania,10784,5740291.0,187.865040,Total Counts,10784.0
4,2010,Injured,Driver,Emilia-Romagna,20034,4459453.0,449.247923,Total Counts,20034.0


In [17]:
# Build interactive chart for casualties by region, metric, and person type
available_regions = chart_data_enhanced[(chart_data_enhanced['Territory'] != 'All Regions') & (~chart_data_enhanced['Territory'].isin(['Provincia Autonoma Bolzano / Bozen', 'Provincia Autonoma Trento']))]['Territory'].unique().tolist()
region_dropdown = alt.binding_select(options=['All Regions'] + sorted(available_regions), name='Select Region: ')
region_selection = alt.selection_point(fields=['Territory'], bind=region_dropdown, value='All Regions')
metric_dropdown = alt.binding_select(options=['Injured + Killed', 'Injured', 'Killed'], name='Select Metric: ')
metric_selection = alt.selection_point(fields=['Result (DESC)'], bind=metric_dropdown, value='Injured + Killed')
display_dropdown = alt.binding_select(options=['Total Counts', 'Per 100,000 Inhabitants'], name='Display As: ')
display_selection = alt.selection_point(fields=['Metric_Type'], bind=display_dropdown, value='Total Counts')
breakdown_checkbox = alt.binding_checkbox(name='Show breakdown by person type: ')
breakdown_selection = alt.selection_point(fields=['show_breakdown'], bind=breakdown_checkbox, value=False)
chart_data_with_total = chart_data_enhanced.copy()
chart_data_with_total['show_breakdown'] = True
aggregated_data = chart_data_enhanced.groupby(['Territory','TIME_PERIOD','Result (DESC)','Metric_Type']).agg({'Value': 'sum','Observation': 'sum','Population': 'first'}).reset_index()
aggregated_data['Person class'] = 'Total'
aggregated_data['show_breakdown'] = False
combined_chart_data = pd.concat([chart_data_with_total, aggregated_data], ignore_index=True)
combined_chart_data['Display_Category'] = combined_chart_data.apply(lambda row: row['Person class'] if row['show_breakdown'] else 'All Person Types', axis=1)
base_encode = {
    'x': alt.X('TIME_PERIOD:O', title='Year', axis=alt.Axis(labelAngle=0)),
    'y': alt.Y('Value:Q', title='Number of People', stack='zero'),
    'color': alt.Color('Display_Category:N', title='Person Type', scale=alt.Scale(domain=['Driver', 'Passenger', 'Pedestrian', 'All Person Types'], range=['#4169E1', '#FF6B6B', '#2ECC71', '#7B68EE'])),
    'opacity': alt.condition(alt.datum.show_breakdown == True, alt.value(1), alt.value(0.85))
}
chart_total_counts = alt.Chart(combined_chart_data).mark_bar().encode(**base_encode, tooltip=[alt.Tooltip('TIME_PERIOD:O', title='Year'),alt.Tooltip('Person class:N', title='Person Type'),alt.Tooltip('Result (DESC):N', title='Metric'),alt.Tooltip('Territory:N', title='Region'),alt.Tooltip('Metric_Type:N', title='Display Type'),alt.Tooltip('Value:Q', title='Count', format=',.0f')]).transform_filter(alt.datum.Metric_Type == 'Total Counts')
chart_per_100k = alt.Chart(combined_chart_data).mark_bar().encode(**base_encode, tooltip=[alt.Tooltip('TIME_PERIOD:O', title='Year'),alt.Tooltip('Person class:N', title='Person Type'),alt.Tooltip('Result (DESC):N', title='Metric'),alt.Tooltip('Territory:N', title='Region'),alt.Tooltip('Metric_Type:N', title='Display Type'),alt.Tooltip('Value:Q', title='Rate per 100k', format=',.2f'),alt.Tooltip('Observation:Q', title='Actual Count', format=',.0f'),alt.Tooltip('Population:Q', title='Population', format=',.0f')]).transform_filter(alt.datum.Metric_Type == 'Per 100,000 Inhabitants')
chart_enhanced = alt.layer(chart_total_counts, chart_per_100k).add_params(region_selection, metric_selection, display_selection, breakdown_selection).transform_filter(region_selection).transform_filter(metric_selection).transform_filter(display_selection).transform_filter(breakdown_selection).properties(width=700, height=400, title={'text': 'Road Accident Casualties in Italy', 'subtitle': 'Source: IstatData (https://esploradati.istat.it/databrowser/#/en).', 'anchor': 'start'})
chart_enhanced

alt.LayerChart(...)

In [18]:
# Save chart specification to JSON
chart_enhanced.save('../graphs/italy_road_accident_casualties_chart.json')